# NDC + Active Drug Lookup

In [22]:
import pandas as pd
import requests

ndc = '00008084381'
url = f"http://localhost:4000/REST/ndcproperties.json?id={ndc}"

response = requests.get(url)


if response.status_code == 200:
    json_data = response.json()
    ndc_properties = json_data['ndcPropertyList']['ndcProperty']
    
    # Normalize just the main level first
    df_base = pd.json_normalize(ndc_properties)
    
    # Extract packaging
    df_base['packaging'] = df_base['packagingList.packaging'].apply(
        lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
    )
    
    # Now normalize properties
    df_properties = pd.json_normalize(
        ndc_properties,
        record_path=['propertyConceptList', 'propertyConcept'],
        meta=['ndcItem', 'rxcui', 'source'],
        errors='ignore'
    )
    
    # Pivot
    df_wide = df_properties.pivot_table(
        index=['ndcItem', 'rxcui', 'source'],
        columns='propName',
        values='propValue',
        aggfunc='first'
    ).reset_index()
    
    # Merge with packaging
    df_final = df_wide.merge(
        df_base[['ndcItem', 'rxcui', 'source', 'packaging']], 
        on=['ndcItem', 'rxcui', 'source']
    )
    
    display(df_final)

,ndcItem,rxcui,source,COLOR,COLORTEXT,IMPRINT_CODE,LABELER,LABEL_TYPE,MARKETING_CATEGORY,MARKETING_EFFECTIVE_TIME_LOW,MARKETING_STATUS,NDA,SCORE,SHAPE,SHAPETEXT,SIZE,packaging
0,00008084381,251872,FDA,NaN,NaN,NaN,"Wyeth Pharmaceuticals LLC, a subsidiary of Pfi...",HUMAN PRESCRIPTION DRUG,NDA,20000501,ACTIVE,NDA020987,NaN,NaN,NaN,NaN,"90 TABLET, DELAYED RELEASE in 1 BOTTLE (0008-0..."
1,00008084381,352125,Hybrid,C48330,YELLOW(YELLOW),P;20,"Wyeth Pharmaceuticals LLC, a subsidiary of Pfi...",HUMAN PRESCRIPTION DRUG,NDA,20000501,ACTIVE,NDA020987,1,C48345,BICONVEX,9 mm,"90 TABLET, DELAYED RELEASE in 1 BOTTLE (0008-0..."


In [23]:
rxcui = df_final.copy()

rxcui = rxcui.rxcui
rxcui = rxcui.astype("string")[0]

display(rxcui)

'251872'

## Obtain Active Drug

In [26]:
active_url = f"http://localhost:4000/REST/rxcui/{rxcui}/active.json"

response = requests.get(active_url)


if response.status_code == 200:
    json_data = response.json()
response = json_data['minConceptGroup']['minConcept']

df_props = pd.json_normalize(
    response
)

name = df_props['name'][0]

# display(df_props['name'])

# print(name)
print(f"Active Drug Name:\n{name}")

Active Drug Name:
pantoprazole 20 MG Delayed Release Oral Tablet
